# Chapter 8 — Mixed Precision Training
**Math for ML Engineers** · companion notebook

This notebook mirrors §8.9 (AMP / bfloat16 / loss scaling). No GPU required — we simulate fp16 and bf16 behaviour using NumPy to make the overflow and underflow dynamics visible without needing real autocast.

**Key numbers to know cold:** fp16 max ≈ 65 504; bf16 max ≈ 3.4 × 10³⁸.

In [ ]:
import numpy as np
import struct
import matplotlib.pyplot as plt


## fp16 vs bf16 — bit layout

Both use 16 bits but allocate them differently:

| Format | Sign | Exponent | Mantissa | Max finite |
|--------|------|----------|----------|------------|
| fp16   | 1    | 5        | 10       | 65 504 ≈ 2¹⁶ |
| bf16   | 1    | 8        | 7        | ≈ 3.4 × 10³⁸ (same as fp32) |

The fp16 exponent field has only 5 bits → max representable exponent is 15 → max value is $(1 + (1-2^{-10})) \times 2^{15} = 65504$.

Attention logits for a single head can reach $\|q\|\|k\| \sim \sqrt{d_k}\cdot\sqrt{d_k} = d_k$. For $d_k = 128$ that is fine; but unnormalised logits before the $1/\sqrt{d_k}$ scale, or with a wrong scale (Failure Mode 1 in Appendix E), can push past 65 504.

In [ ]:
fp16_max  = np.float16(65504)
bf16_approx_max = np.float32(3.4e38)   # same exponent range as fp32

print(f'fp16 max:  {fp16_max}')
print(f'fp32 max:  {np.finfo(np.float32).max:.3e}')
print()

# Overflow demo
x = np.float16(60000)
print(f'fp16: 60000 * 1.2 = {np.float16(60000) * np.float16(1.2)}  (overflow → inf)')
print(f'fp32: 60000 * 1.2 = {np.float32(60000) * np.float32(1.2)}  (fine)')


## Loss scaling: the fp16 underflow problem

Training in fp16 also suffers from **gradient underflow**: gradients small enough to round to zero in fp16 ($< 2^{-24} \approx 6 \times 10^{-8}$).

**Loss scaling** multiplies the loss by a large scalar $S$ before the backward pass, shifting gradients up by $S$ so they survive fp16.  After the backward pass, gradients are divided by $S$ before the optimizer step.

$$\mathcal{L}_{\text{scaled}} = S \cdot \mathcal{L}, \quad g_{\text{true}} = g_{\text{scaled}} / S$$

**Dynamic scaling**: start $S = 2^{15}$; halve it on overflow (inf/nan); double it every 2 000 consecutive clean steps.

In [ ]:
class DynamicLossScaler:
    def __init__(self, init_scale=2**15, growth_factor=2.0, backoff_factor=0.5,
                 growth_interval=2000):
        self.scale = init_scale
        self.growth_factor = growth_factor
        self.backoff_factor = backoff_factor
        self.growth_interval = growth_interval
        self._steps_since_overflow = 0

    def scale_loss(self, loss):
        return loss * self.scale

    def unscale_and_step(self, grad):
        '''Returns (unscaled_grad, skipped) where skipped=True means overflow detected.'''
        if np.any(~np.isfinite(grad)):
            self.scale *= self.backoff_factor
            self._steps_since_overflow = 0
            return grad, True
        self._steps_since_overflow += 1
        if self._steps_since_overflow >= self.growth_interval:
            self.scale *= self.growth_factor
            self._steps_since_overflow = 0
        return grad / self.scale, False


# Simulate 100 steps with occasional synthetic overflow
rng = np.random.default_rng(42)
scaler = DynamicLossScaler()
scale_history = [scaler.scale]

for step in range(500):
    # Synthetic grad: mostly finite, rarely overflows
    grad = rng.normal(0, 1e-4, size=4)
    grad_scaled = grad * scaler.scale
    # Simulate fp16 overflow once every ~100 steps
    if rng.random() < 0.01:
        grad_scaled[0] = np.inf
    _, skipped = scaler.unscale_and_step(grad_scaled)
    scale_history.append(scaler.scale)

plt.figure(figsize=(8, 3))
plt.plot(scale_history)
plt.yscale('log', base=2)
plt.xlabel('Training step'); plt.ylabel('Loss scale $S$ (log₂)')
plt.title('Dynamic loss scale trajectory — overflows cause backoff, clean runs cause growth')
plt.tight_layout(); plt.show()


## Why bf16 doesn't need loss scaling

bf16 has 8 exponent bits (same as fp32) so its representable range is $[\approx 10^{-38}, \approx 3.4 \times 10^{38}]$.  Typical gradient magnitudes ($10^{-4}$–$10^{-1}$) are comfortably in range; only the mantissa precision (7 bits, vs 23 for fp32) is reduced.

The practical rule (§8.9): **always prefer bf16** for Transformer training.  Use fp16 only when hardware doesn't support bf16 (older GPUs before Ampere).

In [ ]:
# Precision comparison: rounding error in fp16 vs simulated bf16
vals = np.logspace(-5, 0, 200, dtype=np.float32)
err_fp16 = np.abs(vals.astype(np.float16).astype(np.float32) - vals) / (vals + 1e-30)
# bf16 has 7 mantissa bits → quantise to 7-bit mantissa simulation
def to_bf16_approx(x):
    bits = np.frombuffer(x.astype(np.float32).tobytes(), dtype=np.uint32)
    bits = (bits & 0xFFFF0000).view(np.float32)  # truncate lower 16 mantissa bits
    return bits
err_bf16 = np.abs(to_bf16_approx(vals) - vals) / (vals + 1e-30)

plt.figure(figsize=(8, 3))
plt.loglog(vals, err_fp16, 'C3', lw=1.2, label='fp16 relative error')
plt.loglog(vals, err_bf16, 'C2', lw=1.2, label='bf16 relative error (simulated)')
plt.axvline(65504, color='C3', ls='--', alpha=0.7, label='fp16 overflow threshold')
plt.xlabel('Value magnitude'); plt.ylabel('Relative rounding error')
plt.title('Rounding error: fp16 vs bf16')
plt.legend(); plt.tight_layout(); plt.show()


## Exercise

1. The dynamic scaler above has `growth_interval=2000`.  What happens if you set it to `200`?  Sketch the effect on the scale trajectory and explain using the exponential moving-average argument from §5.5 (Adam second moment).
2. A model trains in fp16 and consistently overflows at step ~5 000.  List three possible root causes (hint: §8.9, Failure 9 in Appendix E).